# XOR 문제
XOR 문제는, 하나의 퍼셉트론은 AND, OR (초평면으로 나타낼 수 있음.)는 해결이 가능하지만, XOR 문제는 해결이 불가능 하다. 이를 해결하는 방식을 알아보자.

## 등장 (Perceptrons-Minsky&Papert, 1969)
1969년 민스키와 파퍼트의 논문 퍼셉트론에서 해당 문제를 언급하였다. 
퍼셉트론 단일 뉴런의 다음 식은 n차원 공간의 초평면을 나타내는 식과 같다.
$$z=w_1x_1+w_2x_2+...w_nx_n+b=\mathbf{w}^T+\mathbf{b}$$
$$w = 법선벡터$$
$$b = 평행이동 성분$$
이 XOR 문제는 2차원에서 불가능한 것을 바로 알 수 있다.
<T.B.D>

## AND, OR, NAND GATE
퍼셉트론으로 2차원 공간의 AND, OR, NAND 게이트를 표현하려면 다음과 같다.
$$AND) w=(1,1), b=-1.5$$
$$OR) w=(1,1), b=-0.5$$
$$NAND) w=(-1,-1), b=1.5$$
이와 같이 표현이 되는 이유는 최종 출력은 z값의 부호에 의해 결정되기 때문이다. (Step 함수 출력)
즉, AND는 입력이 둘다 1일때만 z값 출력이 0.5가 나오고 이는 1로 출력된다.
OR는 둘중 하나만 1이여도, z값 출력이 0.5가 나오고 이는 1로 출력된다.
NAND는 둘다 0일때만 z값출력이 1.5가 나오고 이는 1로 출력된다.

## 층을 쌓아서 XOR을 해결하는 방법
XOR을 해결하기 위해서는 직선을 두개 그어서 구분하거나, 곡선을 통해 구분해야한다.

In [ ]:
# 가중치를 두개 가지는 최상위 퍼셉트론 클래스 정의
import torch
class Perceptrons:
    def __init__(self, input_dim=2): # 퍼셉트론 생성 시에는 입력 크기를 지정한다.
        self.w = torch.randn(input_dim)
        self.b = torch.randn(1)

    def step(self, z): # Step 함수 정의
        return (z >= 0).float()

    def predict(self, x):
        return self.step(x @ self.w + self.b)
        
    def train(self, x, y, /, epochs=100, lr=0.1): # 스텝 만큼의 학습 호출
        for ep in range(epochs): # 에포크마다 반복함
            errors = 0
            for x, y in zip(x, y):
                err = y - self.predict(x)
                if err != 0:
                    self.w += lr * err * x
                    self.b += lr * err
                    errors += 1
            if errors == 0: # 에러가 없는 에포크일때 에포크 수 반환, 학습 종료
                return ep + 1
        return None

class ANDGate(Perceptrons):
    def __init__(self):
        self.w = torch.tensor([1., 1.])
        self.b = torch.tensor([-1.5])

class ORGate(Perceptrons):
    def __init__(self):
        self.w = torch.tensor([1., 1.])
        self.b = torch.tensor([-0.5])
class NANDGate(Perceptrons):
    def __init__(self):
        self.w = torch.tensor([-1., -1.])
        self.b = torch.tensor([1.5])

In [20]:
# 각 게이트 출력 확인
x = torch.tensor([[1., 1.], [0., 1.], [1., 0.], [0., 0.]])

and_gate = ANDGate()
or_gate = ORGate()
nand_gate = NANDGate()

print(f"AND Gate, expected: [1., 0., 0., 0.], result: {and_gate.predict(x)}")
print(f"OR Gate, expected: [1., 1., 1., 0.], result: {or_gate.predict(x)}")
print(f"NAND Gate, expected: [0., 1., 1., 1.], result: {nand_gate.predict(x)}")

AND Gate, expected: [1., 0., 0., 0.], result: tensor([1., 0., 0., 0.])
OR Gate, expected: [1., 1., 1., 0.], result: tensor([1., 1., 1., 0.])
NAND Gate, expected: [0., 1., 1., 1.], result: tensor([0., 1., 1., 1.])


In [25]:
# 3개를 통과시켜서 XOR 문제가 해결되는지 확인하기
# x1, x2 ─┬─> OR   ──┐
#         │          ├─> AND ─> XOR
#         └─> NAND ──┘
s1 = or_gate.predict(x)
print(s1)
s2 = nand_gate.predict(x)
print(s2)
h = torch.stack([s1, s2], dim=1) # s1, s2의 출력을 열로 쌓기
print(h)

print(f"XOR, expected: [0., 1., 1., 0.], result: {and_gate.predict(h)}")

tensor([1., 1., 1., 0.])
tensor([0., 1., 1., 1.])
tensor([[1., 0.],
        [1., 1.],
        [1., 1.],
        [0., 1.]])
XOR, expected: [0., 1., 1., 0.], result: tensor([0., 1., 1., 0.])
